# Estimate Transformer Parameters, Memory and FLOPs

**Difficulty**: 🟡 Medium

### Problem Statement

"Can we fully fine-tune a 7B model on 8×80 GB GPUs?" "How much KV cache does a 32k-token
context cost?" "How many GPU-hours does pre-training take?" Research-engineer and ML-infra
interviews expect you to answer these in a few minutes from a model config, and real capacity
planning starts the same way. Everything reduces to four counts:

- **Parameters** $N$ — from the architecture (embedding, attention, MLP, norms, LM head).
- **Training memory** — bytes per parameter for weights, gradients and optimizer state
  (activations come on top and depend on batch and sequence length).
- **KV cache** — the keys and values every layer keeps for every past token during generation.
- **Training compute** — $C \approx 6ND$ FLOPs for $D$ training tokens: $2N$ per token forward
  (one multiply-add per parameter) and $4N$ backward (gradients with respect to both the
  activations and the weights).

You will implement each as a small function and check it against real PyTorch objects: a model
built from `nn` modules, real K/V tensors, real optimizer state, and PyTorch's FLOP counter.

---

### Requirements

1. **`transformer_params(vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff,
   tied_embeddings=True, gated_mlp=True, bias=False, norm="rmsnorm") -> int`** — the exact
   parameter count of a Llama-style decoder-only transformer:
   - `head_dim = d_model // n_heads` (assume `d_model % n_heads == 0`, `n_heads % n_kv_heads == 0`).
   - Token embedding: `vocab_size × d_model`. No positional-embedding parameters (RoPE has none).
   - Each of the `n_layers` blocks:
     - a norm before attention; `q_proj`: `d_model → n_heads·head_dim`; `k_proj` and `v_proj`:
       `d_model → n_kv_heads·head_dim` (grouped-query attention — MHA when `n_kv_heads == n_heads`,
       MQA when it is 1); `o_proj`: `n_heads·head_dim → d_model`;
     - a norm before the MLP; gated (SwiGLU) MLP: `gate_proj` and `up_proj` `d_model → d_ff`,
       `down_proj` `d_ff → d_model`; plain MLP (`gated_mlp=False`): only `up_proj` and `down_proj`.
   - `bias=True`: every `Linear` listed above has a bias (one value per output feature).
   - Norms: `"rmsnorm"` has a weight only (`d_model` values); `"layernorm"` has weight and bias
     (`2·d_model`). Anything else → `ValueError`.
   - A final norm after the last block, then the LM head `d_model → vocab_size` **without bias**,
     which shares the embedding matrix when `tied_embeddings=True` (count it once).
2. **`training_memory_gb(n_params, precision="bf16-mixed", optimizer="adamw") -> float`** — decimal
   gigabytes (bytes / 1e9) of weights + gradients + optimizer state, excluding activations and
   temporary buffers, using this standard accounting (bytes per parameter):

   | `precision` | weights | gradients | fp32 master weights |
   |---|---|---|---|
   | `"fp32"` | 4 | 4 | – |
   | `"bf16-mixed"` | 2 (bf16) | 2 (bf16) | 4 |
   | `"fp16-mixed"` | 2 (fp16) | 2 (fp16) | 4 |

   | `optimizer` | fp32 state per parameter | bytes |
   |---|---|---|
   | `"adamw"`, `"adam"` | first moment $m$, second moment $v$ | 8 |
   | `"sgd-momentum"` | momentum buffer | 4 |
   | `"sgd"` | none | 0 |

   So AdamW costs **16 bytes/parameter** in both fp32 and mixed precision (the classic
   "2 + 2 + 12" of the ZeRO paper). Unknown `precision` or `optimizer` → `ValueError`.
3. **`kv_cache_bytes(batch, seq_len, n_layers, n_kv_heads, head_dim, bytes_per_elem=2) -> int`** —
   bytes needed to cache K and V for every layer and token: one K and one V tensor of shape
   `(batch, n_kv_heads, seq_len, head_dim)` per layer.
4. **`training_flops(n_params, n_tokens) -> float`** — the standard estimate $6\,N\,D$.

---

### Constraints

- ✅ Plain Python arithmetic — these are formulas you should be able to write on a whiteboard.
- ❌ Don't instantiate a model inside `transformer_params` (the tests build one to check *you*,
  and a real 8B model would not fit in memory anyway).
- ❌ No `torch.utils.flop_counter` inside `training_flops`.

---

<details>
  <summary>💡 Hint</summary>

  - Write the count of one `nn.Linear(fan_in, fan_out, bias=b)` as a helper first.
  - With GQA, which projections shrink? Which ones keep `d_model` outputs?
  - In PyTorch, tying is `lm_head.weight = embed.weight`; `model.parameters()` yields a shared
    tensor once. How many vocabulary-sized matrices does an untied model have?
  - For memory, list *what tensors exist* during a mixed-precision step and how many bytes each
    element takes. Which of them does `torch.optim.AdamW` create?
  - For FLOPs: a matmul `(T, a) @ (a, b)` costs $2Tab$. In backward, which two matmuls does each
    `Linear` run?

</details>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.flop_counter import FlopCounterMode

In [ ]:
# Real model configs (from the published Hugging Face config.json files)
LLAMA2_7B = dict(vocab_size=32000, d_model=4096, n_layers=32, n_heads=32, n_kv_heads=32,
                 d_ff=11008, tied_embeddings=False)
LLAMA3_8B = dict(vocab_size=128256, d_model=4096, n_layers=32, n_heads=32, n_kv_heads=8,
                 d_ff=14336, tied_embeddings=False)


class Block(nn.Module):
    """One Llama-style decoder block — parameters only (no forward needed to count them)."""

    def __init__(self, d, n_heads, n_kv, d_ff, gated=True, bias=False, norm="rmsnorm"):
        super().__init__()
        hd = d // n_heads
        Norm = nn.RMSNorm if norm == "rmsnorm" else nn.LayerNorm
        self.attn_norm, self.mlp_norm = Norm(d), Norm(d)
        self.q_proj = nn.Linear(d, n_heads * hd, bias=bias)
        self.k_proj = nn.Linear(d, n_kv * hd, bias=bias)
        self.v_proj = nn.Linear(d, n_kv * hd, bias=bias)
        self.o_proj = nn.Linear(n_heads * hd, d, bias=bias)
        self.gate_proj = nn.Linear(d, d_ff, bias=bias) if gated else None
        self.up_proj = nn.Linear(d, d_ff, bias=bias)
        self.down_proj = nn.Linear(d_ff, d, bias=bias)


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff,
                 tied_embeddings=True, gated_mlp=True, bias=False, norm="rmsnorm"):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(Block(d_model, n_heads, n_kv_heads, d_ff, gated_mlp, bias, norm)
                                    for _ in range(n_layers))
        self.norm = nn.RMSNorm(d_model) if norm == "rmsnorm" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tied_embeddings:
            self.lm_head.weight = self.embed_tokens.weight


def count_real(**cfg):
    """Build the model on the meta device (no memory allocated) and count its parameters."""
    with torch.device("meta"):
        model = Decoder(**cfg)
    return sum(p.numel() for p in model.parameters())   # tied weights are yielded once


print(f"Llama-2-7B, built for real: {count_real(**LLAMA2_7B):,} parameters")

In [ ]:
def transformer_params(vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff,
                       tied_embeddings=True, gated_mlp=True, bias=False, norm="rmsnorm"):
    """Exact parameter count of a Llama-style decoder-only transformer.

    Args:
        vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff: architecture sizes.
        tied_embeddings: LM head shares the token-embedding matrix.
        gated_mlp: SwiGLU MLP with 3 matrices (gate, up, down) instead of 2 (up, down).
        bias: every attention/MLP Linear has a bias (the LM head never does).
        norm: "rmsnorm" (weight) or "layernorm" (weight + bias).

    Returns:
        int — number of parameters.
    """
    # TODO: head_dim, and a helper for one Linear's parameter count
    # TODO: per layer: q/k/v/o projections (k, v have n_kv_heads * head_dim outputs), MLP, 2 norms
    # TODO: total = embedding + n_layers * per_layer + final norm + LM head (unless tied)
    raise NotImplementedError

In [ ]:
def training_memory_gb(n_params, precision="bf16-mixed", optimizer="adamw"):
    """Weights + gradients + optimizer state, in decimal GB (bytes / 1e9). No activations.

    Args:
        n_params: number of parameters.
        precision: "fp32", "bf16-mixed" or "fp16-mixed".
        optimizer: "adamw", "adam", "sgd-momentum" or "sgd".

    Returns:
        float — gigabytes.
    """
    # TODO: bytes/param for weights, grads and the fp32 master copy (see the table)
    # TODO: + optimizer-state bytes/param; raise ValueError on unknown names
    raise NotImplementedError

In [ ]:
def kv_cache_bytes(batch, seq_len, n_layers, n_kv_heads, head_dim, bytes_per_elem=2):
    """Bytes of the K and V cache for every layer and position.

    Returns:
        int — bytes.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def training_flops(n_params, n_tokens):
    """Approximate training compute: forward + backward over n_tokens tokens.

    Returns:
        float — FLOPs.
    """
    # TODO: 2N per token forward, 4N per token backward
    raise NotImplementedError

In [ ]:
# Test 1: parameter counts agree with models built from nn modules
configs = [
    dict(LLAMA2_7B),                                                       # MHA, untied
    dict(LLAMA3_8B),                                                       # GQA (8 KV heads)
    dict(vocab_size=101, d_model=64, n_layers=3, n_heads=8, n_kv_heads=2, d_ff=172),   # tied
    dict(vocab_size=50, d_model=48, n_layers=2, n_heads=6, n_kv_heads=1, d_ff=192,
         gated_mlp=False, bias=True, norm="layernorm"),                    # MQA, GPT-style MLP
]
for cfg in configs:
    got, want = transformer_params(**cfg), count_real(**cfg)
    assert got == want, f"{cfg}: got {got:,}, the real model has {want:,} (off by {got - want:,})"
assert transformer_params(**LLAMA2_7B) == 6_738_415_616
assert transformer_params(**LLAMA3_8B) == 8_030_261_248
print(f"Test 1 passed: Llama-2-7B {transformer_params(**LLAMA2_7B) / 1e9:.2f}B, "
      f"Llama-3-8B {transformer_params(**LLAMA3_8B) / 1e9:.2f}B")

In [ ]:
# Test 2: training memory — the 16 bytes/param rule, and real AdamW state
n = transformer_params(**LLAMA2_7B)
assert abs(training_memory_gb(n, "bf16-mixed", "adamw") - n * 16 / 1e9) < 1e-6
assert abs(training_memory_gb(n, "fp32", "adamw") - n * 16 / 1e9) < 1e-6
assert abs(training_memory_gb(n, "fp32", "sgd") - n * 8 / 1e9) < 1e-6
assert abs(training_memory_gb(n, "bf16-mixed", "sgd-momentum") - n * 12 / 1e9) < 1e-6

lin = nn.Linear(256, 128)                         # fp32 model, AdamW: count the real tensors
opt = torch.optim.AdamW(lin.parameters())
lin(torch.randn(4, 256)).sum().backward()
opt.step()
P = sum(p.numel() for p in lin.parameters())
held = sum(t.numel() * t.element_size() for p in lin.parameters()
           for t in [p, p.grad, *opt.state[p].values()] if t.shape == p.shape)
assert abs(training_memory_gb(P, "fp32", "adamw") - held / 1e9) < 1e-15
try:
    training_memory_gb(n, "int3")
    raise AssertionError("unknown precision should raise ValueError")
except ValueError:
    pass
print(f"Test 2 passed: full fine-tuning Llama-2-7B with AdamW needs "
      f"{training_memory_gb(n):.1f} GB before activations")

In [ ]:
# Test 3: KV cache size == bytes of the K/V tensors a decoder actually allocates
batch, seq, layers, kv_heads, head_dim = 2, 10, 3, 2, 8
cache = [(torch.empty(batch, kv_heads, seq, head_dim, dtype=torch.bfloat16),
          torch.empty(batch, kv_heads, seq, head_dim, dtype=torch.bfloat16)) for _ in range(layers)]
real = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in cache)
assert kv_cache_bytes(batch, seq, layers, kv_heads, head_dim, bytes_per_elem=2) == real
assert kv_cache_bytes(batch, seq, layers, kv_heads, head_dim, bytes_per_elem=4) == 2 * real
llama2 = kv_cache_bytes(1, 4096, 32, 32, 128)
assert llama2 == 2 ** 31
print(f"Test 3 passed: Llama-2-7B, 4k context, batch 1, bf16: {llama2 / 1e9:.2f} GB of KV cache")

In [ ]:
# Test 4: 6·N·D matches PyTorch's FLOP counter on a forward + backward pass
net = nn.Sequential(nn.Linear(32, 64, bias=False), nn.Linear(64, 48, bias=False),
                    nn.Linear(48, 16, bias=False))
x = torch.randn(20, 32, requires_grad=True)       # 20 tokens
with FlopCounterMode(display=False) as counter:
    net(x).sum().backward()
P = sum(p.numel() for p in net.parameters())
assert training_flops(P, 20) == counter.get_total_flops(), \
    f"6ND = {training_flops(P, 20)}, measured {counter.get_total_flops()}"
print(f"Test 4 passed: {counter.get_total_flops():,} FLOPs measured = 6 x {P} params x 20 tokens")
print(f"Llama-2-7B on 2T tokens: {training_flops(transformer_params(**LLAMA2_7B), 2e12):.2e} FLOPs")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("memory-flops-estimation", transformer_params=transformer_params,
      training_memory_gb=training_memory_gb, kv_cache_bytes=kv_cache_bytes,
      training_flops=training_flops)